In [4]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, precision_score,
                              recall_score, f1_score, accuracy_score)
import shap

df = pd.read_csv('../data/processed/02_features.csv')

# ── FEATURES ──
FEATURES = [
    'Stock Quantity', 'days_to_expiry', 'Price',
    'stock_pressure', 'expiry_urgency', 'risk_score',
    'is_expired', 'high_stock_flag', 'near_expiry_flag',
    'price_tier_enc', 'category_enc', 'warranty_enc',
    'financial_exposure', 'overhang_ratio'
]

TARGET = 'is_at_risk'

X = df[FEATURES]
y = df[TARGET]

# ── SPLIT ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# ── BASELINE: Logistic Regression ──
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
lr_prob = lr.predict_proba(X_test)[:,1]

lr_metrics = {
    'AUC-ROC':   round(roc_auc_score(y_test, lr_prob), 4),
    'Precision': round(precision_score(y_test, lr_pred), 4),
    'Recall':    round(recall_score(y_test, lr_pred), 4),
    'F1-Score':  round(f1_score(y_test, lr_pred), 4),
    'Accuracy':  round(accuracy_score(y_test, lr_pred), 4),
}
print("\nLogistic Regression:", lr_metrics)

# ── MAIN MODEL: Random Forest ──
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:,1]

rf_metrics = {
    'AUC-ROC':   round(roc_auc_score(y_test, rf_prob), 4),
    'Precision': round(precision_score(y_test, rf_pred), 4),
    'Recall':    round(recall_score(y_test, rf_pred), 4),
    'F1-Score':  round(f1_score(y_test, rf_pred), 4),
    'Accuracy':  round(accuracy_score(y_test, rf_pred), 4),
}
print("Random Forest:", rf_metrics)

# ── SHAP FEATURE IMPORTANCE ──
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

shap_importance = pd.DataFrame({
    'feature': FEATURES,
    'importance': np.abs(shap_values[1]).mean(axis=0)
}).sort_values('importance', ascending=False)

shap_importance['pct'] = (
    shap_importance['importance'] / shap_importance['importance'].sum() * 100
).round(1)

print("\nTop 10 features:")
print(shap_importance.head(10))

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values[1], X_test, feature_names=FEATURES,
                  show=False, plot_type='bar')
plt.tight_layout()
plt.savefig('../reports/figures/05_shap_importance.png', bbox_inches='tight')
plt.close()
print("SHAP plot saved.")

# ── DASHBOARD EXPORT ──

# safer risk breakdown (handles missing categories)
risk_breakdown = df['Risk_Level'].value_counts().reindex(
    ['Critical','High','Medium','Low'], fill_value=0
).to_dict()

dashboard_data = {
    'kpis': {
        'total_skus': len(df),
        'avg_stock_qty': round(df['Stock Quantity'].mean(), 1),

        # FIXED: use days_to_expiry instead of Days_to_Expiry
        'dual_risk_count': int(df[
            (df['Stock Quantity'] > 75) & (df['days_to_expiry'] < 180)
        ].shape[0]),

        'capital_at_risk': round(
            df[df['is_at_risk']==1]['financial_exposure'].sum(), 0
        ),

        # FIXED: <= 0 instead of == 0
        'write_off_rate': round(
            (df['days_to_expiry'] <= 0).mean() * 100, 1
        )
    },

    'risk_breakdown': risk_breakdown,

    'model_logistic': lr_metrics,
    'model_rf': rf_metrics,

    'feature_importance': shap_importance.head(10)[['feature','pct']].to_dict('records'),

    'stock_distribution': df['stock_bucket'].value_counts().sort_index().to_dict(),
    'expiry_distribution': df['expiry_bucket'].value_counts().sort_index().to_dict(),

    'price_by_category': df.groupby('Product Category')['Price'].mean().round(2).to_dict(),
}

with open('../outputs/dashboard_data.json', 'w') as f:
    json.dump(dashboard_data, f, indent=2)

print("\ndashboard_data.json saved to outputs/")


Train: (8000, 14), Test: (2000, 14)

Logistic Regression: {'AUC-ROC': 1.0, 'Precision': 1.0, 'Recall': 1.0, 'F1-Score': 1.0, 'Accuracy': 1.0}
Random Forest: {'AUC-ROC': 1.0, 'Precision': 1.0, 'Recall': 1.0, 'F1-Score': 1.0, 'Accuracy': 1.0}

Top 10 features:
               feature  importance   pct
7      high_stock_flag    0.087406  22.8
0       Stock Quantity    0.081431  21.3
13      overhang_ratio    0.075783  19.8
5           risk_score    0.065419  17.1
3       stock_pressure    0.063599  16.6
12  financial_exposure    0.007792   2.0
2                Price    0.000809   0.2
9       price_tier_enc    0.000236   0.1
11        warranty_enc    0.000139   0.0
1       days_to_expiry    0.000097   0.0


The figure layout has changed to tight


SHAP plot saved.

dashboard_data.json saved to outputs/
